# 02 - Editar medios propios y renderizar con Remotion en R2 privado

Guarda en Colab Secrets: `API_BASE_URL` (HTTPS publico), `WORKER_API_TOKEN`, `S3_ENDPOINT_URL`, `S3_BUCKET`, `S3_ACCESS_KEY_ID` y `S3_SECRET_ACCESS_KEY`. El worker descarga en privado los medios seleccionados desde R2, compone imagenes/videos propios, guion y voz, y sube el resultado. Ejecuta este notebook en una sesion de Colab separada del worker de investigacion.

In [ ]:
!apt-get update -qq && apt-get install -y -qq nodejs npm ffmpeg chromium

In [ ]:
%pip install -q boto3 requests

In [ ]:
import os
from google.colab import userdata

REQUIRED = ("API_BASE_URL", "WORKER_API_TOKEN", "S3_ENDPOINT_URL", "S3_BUCKET",
            "S3_ACCESS_KEY_ID", "S3_SECRET_ACCESS_KEY")
for name in REQUIRED:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Configura el secreto {name} en Colab")
    os.environ[name] = value
os.environ["API_BASE_URL"] = os.environ["API_BASE_URL"].rstrip("/")
if not os.environ["API_BASE_URL"].startswith("https://"):
    raise RuntimeError("API_BASE_URL debe ser HTTPS y accesible desde Colab")
API_HEADERS = {"Authorization": f"Bearer {os.environ['WORKER_API_TOKEN']}"}

In [ ]:
import json
import math
import shutil
import threading
import time
import subprocess
from pathlib import Path

import boto3
import requests

PROJECT = Path("/content/marketing-content-os-renderer")
PUBLIC = PROJECT / "public"
SRC = PROJECT / "src"
PUBLIC.mkdir(parents=True, exist_ok=True)
SRC.mkdir(parents=True, exist_ok=True)
(PROJECT / "package.json").write_text(json.dumps({
    "private": True,
    "scripts": {"render": "remotion render src/index.ts MarketingShort"},
    "dependencies": {"@remotion/cli": "4.0.448", "react": "19.0.0", "react-dom": "19.0.0", "remotion": "4.0.448"},
    "devDependencies": {"@types/react": "19.0.0", "typescript": "5.7.3"}
}, indent=2), encoding="utf-8")
(SRC / "index.ts").write_text('import { registerRoot } from "remotion"; import { Root } from "./Root"; registerRoot(Root);', encoding="utf-8")
(SRC / "Root.tsx").write_text('''import React from "react";
import { AbsoluteFill, Audio, Composition, Img, OffthreadVideo, staticFile, useCurrentFrame, useVideoConfig } from "remotion";
type Props = { topic: string; script: string; duration_seconds: number; visual_asset?: { asset_type: "image" | "video"; filename: string } | null };
const Video: React.FC<Props> = ({ topic, script, visual_asset }) => {
  const frame = useCurrentFrame(); const { fps } = useVideoConfig();
  const words = script.split(/\\s+/); const chunk = Math.max(1, Math.ceil(words.length / 8));
  const caption = words.slice(Math.floor(frame / (fps * 4)) * chunk, Math.floor(frame / (fps * 4)) * chunk + chunk).join(" ");
  return <AbsoluteFill style={{background:"linear-gradient(145deg,#0b1220,#123a55)",color:"white",fontFamily:"Arial",padding:90,justifyContent:"space-between"}}>
    {visual_asset?.asset_type === "image" && <Img src={staticFile(visual_asset.filename)} style={{position:"absolute",inset:0,width:"100%",height:"100%",objectFit:"cover"}} />}
    {visual_asset?.asset_type === "video" && <OffthreadVideo src={staticFile(visual_asset.filename)} muted style={{position:"absolute",inset:0,width:"100%",height:"100%",objectFit:"cover"}} />}
    <div style={{fontSize:54,fontWeight:700}}>{topic}</div><div style={{fontSize:48,lineHeight:1.35,textShadow:"0 2px 10px #000"}}>{caption}</div>
    <div style={{fontSize:26,opacity:.8}}>Marketing Content OS</div><Audio src={staticFile("voice.wav")} />
  </AbsoluteFill>;
};
export const Root: React.FC = () => <Composition id="MarketingShort" component={Video} width={1080} height={1920} fps={30} durationInFrames={1050} defaultProps={{topic:"Tema",script:"",duration_seconds:35}} calculateMetadata={({props}) => ({durationInFrames:Math.max(30,Math.ceil(props.duration_seconds*30))})} />;
''', encoding="utf-8")
subprocess.run(["npm", "install", "--no-audit", "--no-fund"], cwd=PROJECT, check=True)
subprocess.run(["npx", "remotion", "browser", "ensure"], cwd=PROJECT, check=True)

In [ ]:
def start_lease_heartbeat(job_id, lease_headers):
    stop = threading.Event()

    def renew_lease():
        while not stop.wait(60):
            try:
                response = requests.post(
                    f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/heartbeat",
                    headers=lease_headers,
                    timeout=30,
                )
                response.raise_for_status()
            except requests.RequestException as error:
                print(f"No se pudo renovar la reserva del trabajo {job_id}: {error}")
                return

    thread = threading.Thread(target=renew_lease, daemon=True)
    thread.start()
    return stop, thread


def report_progress(job_id, lease_headers, stage, message, percent):
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/progress",
        headers=lease_headers,
        json={"stage": stage, "message": message, "percent": percent},
        timeout=30,
    )
    response.raise_for_status()


def process_one_render_job():
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/jobs/next",
        params={"worker": "render"}, headers=API_HEADERS, timeout=30,
    )
    response.raise_for_status()
    job = response.json()
    if job is None:
        print("No hay renders pendientes.")
        return False

    job_id = job["job_id"]
    lease_headers = {**API_HEADERS, "X-Worker-Lease": job["lease_token"]}
    heartbeat_stop, heartbeat_thread = start_lease_heartbeat(job_id, lease_headers)
    try:
        request = job["request"]
        report_progress(job_id, lease_headers, "media_download", "Descargando audio y medios privados de R2", 15)
        if not request.get("script") or not (request.get("audio_object_key") or request.get("audio_url")):
            raise ValueError("El trabajo requiere guion y audio")
        storage = boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT_URL"], region_name="auto",
                               aws_access_key_id=os.environ["S3_ACCESS_KEY_ID"],
                               aws_secret_access_key=os.environ["S3_SECRET_ACCESS_KEY"])
        if request.get("audio_object_key"):
            storage.download_file(os.environ["S3_BUCKET"], request["audio_object_key"], str(PUBLIC / "voice.wav"))
        else:
            audio = requests.get(request["audio_url"], timeout=120)
            audio.raise_for_status()
            (PUBLIC / "voice.wav").write_bytes(audio.content)
        visual_asset = request.get("visual_asset")
        visual_props = None
        if visual_asset:
            suffix = {"image/png": ".png", "image/jpeg": ".jpg", "image/webp": ".webp", "video/mp4": ".mp4"}.get(visual_asset["content_type"])
            if suffix is None:
                raise ValueError("Formato de imagen/video no compatible para renderizar")
            visual_filename = f"background{suffix}"
            storage.download_file(os.environ["S3_BUCKET"], visual_asset["object_key"], str(PUBLIC / visual_filename))
            visual_props = {"asset_type": visual_asset["asset_type"], "filename": visual_filename}
        report_progress(job_id, lease_headers, "media_ready", "Medios listos; preparando edición", 35)
        props = {"topic": request["topic"], "script": request["script"],
                 "duration_seconds": request["duration_seconds"], "visual_asset": visual_props}
        props_path = Path("/content/render-props.json")
        props_path.write_text(json.dumps(props, ensure_ascii=False), encoding="utf-8")
        video_path = Path(f"/content/{job_id}.mp4")
        report_progress(job_id, lease_headers, "rendering", "Editando y renderizando el video", 50)
        subprocess.run([
            "npx", "remotion", "render", "src/index.ts", "MarketingShort", str(video_path),
            f"--props={props_path}", "--codec=h264", "--browser-executable=/usr/bin/chromium",
        ], cwd=PROJECT, check=True)
        if not video_path.is_file() or video_path.stat().st_size == 0:
            raise RuntimeError("Remotion no produjo un MP4 valido")

        report_progress(job_id, lease_headers, "video_storage", "Subiendo video terminado a R2 privado", 85)
        video_key = f"renders/{job_id}.mp4"
        storage.upload_file(str(video_path), os.environ["S3_BUCKET"], video_key,
                            ExtraArgs={"ContentType": "video/mp4"})
        complete = requests.post(
            f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/complete",
            headers=lease_headers, json={"output": {"video_object_key": video_key}}, timeout=60,
        )
        complete.raise_for_status()
        print(f"Video listo: {job_id}")
    except Exception as error:
        failed = requests.post(
            f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/fail",
            headers=lease_headers, json={"error": str(error)[:1900]}, timeout=60,
        )
        failed.raise_for_status()
        raise RuntimeError(f"Trabajo {job_id} marcado como fallido: {error}") from error
    finally:
        heartbeat_stop.set()
        heartbeat_thread.join(timeout=5)
    return True


POLL_INTERVAL_SECONDS = 30
print("Worker activo. Deten la celda para parar de consultar la cola.")
while True:
    try:
        processed = process_one_render_job()
    except Exception as error:
        print(f"Error del worker; volvera a intentarlo en {POLL_INTERVAL_SECONDS} s: {error}")
        time.sleep(POLL_INTERVAL_SECONDS)
        continue
    if not processed:
        time.sleep(POLL_INTERVAL_SECONDS)